In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [2]:
df = pd.read_csv('/kaggle/input/datasets/shivam131019/telecom-churn-dataset/telecom_churn_data.csv')

print(df.shape)
print(df.head())

(99999, 226)
   mobile_number  circle_id  loc_og_t2o_mou  std_og_t2o_mou  loc_ic_t2o_mou  \
0     7000842753        109             0.0             0.0             0.0   
1     7001865778        109             0.0             0.0             0.0   
2     7001625959        109             0.0             0.0             0.0   
3     7001204172        109             0.0             0.0             0.0   
4     7000142493        109             0.0             0.0             0.0   

  last_date_of_month_6 last_date_of_month_7 last_date_of_month_8  \
0            6/30/2014            7/31/2014            8/31/2014   
1            6/30/2014            7/31/2014            8/31/2014   
2            6/30/2014            7/31/2014            8/31/2014   
3            6/30/2014            7/31/2014            8/31/2014   
4            6/30/2014            7/31/2014            8/31/2014   

  last_date_of_month_9   arpu_6  ...  sachet_3g_9  fb_user_6  fb_user_7  \
0            9/30/2014  197.

In [12]:
def create_month_snapshot(df, current_month, next_month):
    
    data = df.copy()

    # --------------------------------------------------
    # High-value customers based on CURRENT month ARPU
    # --------------------------------------------------
    
    arpu_col = f"arpu_{current_month}"
    
    threshold = data[arpu_col].quantile(0.70)
    
    data = data[
        data[arpu_col] >= threshold
    ].copy()

    # --------------------------------------------------
    # Create next-month activity
    # --------------------------------------------------
    
    next_activity = (
        data[f"total_og_mou_{next_month}"].fillna(0)
        + data[f"total_ic_mou_{next_month}"].fillna(0)
        + data[f"total_rech_num_{next_month}"].fillna(0)
        + data[f"vol_2g_mb_{next_month}"].fillna(0)
        + data[f"vol_3g_mb_{next_month}"].fillna(0)
    )

    # 1 = churn, 0 = active
    data["churn"] = (
        next_activity == 0
    ).astype(int)

    # --------------------------------------------------
    # Select CURRENT month columns
    # --------------------------------------------------
    
    current_suffix = f"_{current_month}"

    month_cols = []

    for col in data.columns:

        if col.endswith(current_suffix):
            month_cols.append(col)

    # Columns that do not have a monthly suffix
    # and are valid customer-level variables
    static_cols = [
        "mobile_number",
        "circle_id",
        "aon"
    ]

    static_cols = [
        col for col in static_cols
        if col in data.columns
    ]

    # --------------------------------------------------
    # Special VBC columns
    # --------------------------------------------------

    vbc_mapping = {
        "6": "jun_vbc_3g",
        "7": "jul_vbc_3g",
        "8": "aug_vbc_3g"
    }

    if current_month in vbc_mapping:

        vbc_col = vbc_mapping[current_month]

        if vbc_col in data.columns:
            month_cols.append(vbc_col)

    # --------------------------------------------------
    # Build snapshot
    # --------------------------------------------------

    snapshot = data[
        static_cols + month_cols + ["churn"]
    ].copy()

    # --------------------------------------------------
    # Remove month suffix
    # --------------------------------------------------

    rename_dict = {}

    for col in snapshot.columns:

        if col.endswith(current_suffix):

            new_name = col[
                :-len(current_suffix)
            ]

            rename_dict[col] = new_name

    # Rename VBC column
    if current_month in vbc_mapping:

        vbc_col = vbc_mapping[current_month]

        if vbc_col in snapshot.columns:

            rename_dict[vbc_col] = "vbc_3g"

    snapshot = snapshot.rename(
        columns=rename_dict
    )

    return snapshot, threshold

In [13]:
jun_data, jun_threshold = create_month_snapshot(
    df,
    current_month="6",
    next_month="7"
)

print("June → July")
print("Customers:", len(jun_data))
print("Churn rate:", jun_data["churn"].mean() * 100)

June → July
Customers: 30000
Churn rate: 0.74


In [14]:
jul_data, jul_threshold = create_month_snapshot(
    df,
    current_month="7",
    next_month="8"
)

print("\nJuly → August")
print("Customers:", len(jul_data))
print("Churn rate:", jul_data["churn"].mean() * 100)


July → August
Customers: 30000
Churn rate: 0.62


In [15]:
aug_data, aug_threshold = create_month_snapshot(
    df,
    current_month="8",
    next_month="9"
)

print("\nAugust → September")
print("Customers:", len(aug_data))
print("Churn rate:", aug_data["churn"].mean() * 100)


August → September
Customers: 30000
Churn rate: 0.5033333333333333


In [16]:
monthly_data = pd.concat(
    [
        jun_data,
        jul_data,
        aug_data
    ],
    ignore_index=True
)

print("Combined shape:", monthly_data.shape)

print(
    monthly_data["churn"].value_counts()
)

print(
    monthly_data["churn"]
    .value_counts(normalize=True) * 100
)

Combined shape: (90000, 59)
churn
0    89441
1      559
Name: count, dtype: int64
churn
0    99.378889
1     0.621111
Name: proportion, dtype: float64


In [17]:
X = monthly_data.drop(
    columns=[
        "churn",
        "mobile_number"
    ],
    errors="ignore"
)

y = monthly_data["churn"]

In [18]:
date_cols = [
    col
    for col in X.columns
    if "date" in col.lower()
]

X = X.drop(
    columns=date_cols,
    errors="ignore"
)

In [19]:
categorical_cols = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

X = pd.get_dummies(
    X,
    columns=categorical_cols,
    drop_first=True
)

In [20]:
imputer = SimpleImputer(
    strategy="median"
)

X = pd.DataFrame(
    imputer.fit_transform(X),
    columns=X.columns,
    index=X.index
)


In [21]:
model_features = X.columns.tolist()

print(
    "Number of features:",
    len(model_features)
)

Number of features: 54


In [22]:
groups = monthly_data["mobile_number"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        X,
        y,
        groups=groups
    )
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (71972, 54)
Testing: (18028, 54)


In [25]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train,
    y_train
)

RandomForestClassifier(class_weight='balanced', max_depth=12,
                       min_samples_leaf=5, min_samples_split=10,
                       n_estimators=300, n_jobs=-1, random_state=42)

In [26]:
y_pred = rf_model.predict(X_test)

y_prob = rf_model.predict_proba(
    X_test
)[:, 1]

In [27]:
print(
    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       0.99      1.00      1.00     17903
           1       0.65      0.26      0.37       125

    accuracy                           0.99     18028
   macro avg       0.82      0.63      0.68     18028
weighted avg       0.99      0.99      0.99     18028



In [28]:
print(
    "Accuracy:",
    accuracy_score(y_test, y_pred)
)

print(
    "Precision:",
    precision_score(
        y_test,
        y_pred,
        zero_division=0
    )
)

print(
    "Recall:",
    recall_score(
        y_test,
        y_pred,
        zero_division=0
    )
)

print(
    "F1:",
    f1_score(
        y_test,
        y_pred,
        zero_division=0
    )
)

print(
    "ROC-AUC:",
    roc_auc_score(
        y_test,
        y_prob
    )
)

Accuracy: 0.9938983802973153
Precision: 0.6530612244897959
Recall: 0.256
F1: 0.367816091954023
ROC-AUC: 0.745595263363682


In [29]:
joblib.dump(
    rf_model,
    "/kaggle/working/churn_model_1month.pkl"
)

joblib.dump(
    imputer,
    "/kaggle/working/imputer_1month.pkl"
)

joblib.dump(
    model_features,
    "/kaggle/working/features_1month.pkl"
)

['/kaggle/working/features_1month.pkl']

In [30]:
thresholds = {
    "June": jun_threshold,
    "July": jul_threshold,
    "August": aug_threshold
}

joblib.dump(
    thresholds,
    "/kaggle/working/high_value_thresholds.pkl"
)

['/kaggle/working/high_value_thresholds.pkl']

In [31]:
# Create an August-only test dataset

august_test = df.copy()

# Remove September columns
september_columns = [
    col
    for col in august_test.columns
    if col.endswith("_9")
]

august_test = august_test.drop(
    columns=september_columns,
    errors="ignore"
)

# Save test file
august_test.to_csv(
    "/kaggle/working/august_test.csv",
    index=False
)

print(
    "August test dataset:",
    august_test.shape
)

August test dataset: (99999, 172)


In [32]:
# Actual July -> August churn
# among July high-value customers

july_hv = df[
    df["arpu_7"] >= jul_threshold
].copy()

aug_activity = (
    july_hv["total_og_mou_8"].fillna(0)
    + july_hv["total_ic_mou_8"].fillna(0)
    + july_hv["total_rech_num_8"].fillna(0)
    + july_hv["vol_2g_mb_8"].fillna(0)
    + july_hv["vol_3g_mb_8"].fillna(0)
)

july_hv["actual_churn"] = (
    aug_activity == 0
).astype(int)

actual_churn_rate = (
    july_hv["actual_churn"].mean() * 100
)

print(
    f"Actual August churn: {actual_churn_rate:.2f}%"
)

print(
    f"Customers: {len(july_hv):,}"
)

print(
    july_hv["actual_churn"].value_counts()
)

Actual August churn: 0.62%
Customers: 30,000
actual_churn
0    29814
1      186
Name: count, dtype: int64


In [33]:
model_features = X.columns.tolist()

In [34]:
# Predictions for July -> August
july_data = jul_data.copy()

X_july = july_data.drop(
    columns=["churn", "mobile_number"],
    errors="ignore"
)

y_july = july_data["churn"]

# Remove date columns
date_cols = [
    col for col in X_july.columns
    if "date" in col.lower()
]

X_july = X_july.drop(
    columns=date_cols,
    errors="ignore"
)

# One-hot encode
categorical_cols = X_july.select_dtypes(
    include=["object", "category"]
).columns.tolist()

X_july = pd.get_dummies(
    X_july,
    columns=categorical_cols,
    drop_first=True
)

# Align with training features
for col in model_features:
    if col not in X_july.columns:
        X_july[col] = 0

X_july = X_july[model_features]

# Impute
X_july = pd.DataFrame(
    imputer.transform(X_july),
    columns=model_features,
    index=X_july.index
)

# Predict
july_prob = rf_model.predict_proba(X_july)[:, 1]
july_pred = (july_prob >= 0.5).astype(int)

print("Predicted churn:", july_pred.mean() * 100)
print("Actual churn:", y_july.mean() * 100)

print("\nPredicted classes:")
print(pd.Series(july_pred).value_counts())

print("\nProbability statistics:")
print(pd.Series(july_prob).describe())

Predicted churn: 0.5533333333333333
Actual churn: 0.62

Predicted classes:
0    29834
1      166
Name: count, dtype: int64

Probability statistics:
count    30000.000000
mean         0.134562
std          0.093024
min          0.000565
25%          0.064008
50%          0.116636
75%          0.185388
max          0.994428
dtype: float64


In [35]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_july,
    july_pred
)

print(cm)

[[29788    26]
 [   46   140]]


In [36]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_july,
        july_pred,
        zero_division=0
    )
)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     29814
           1       0.84      0.75      0.80       186

    accuracy                           1.00     30000
   macro avg       0.92      0.88      0.90     30000
weighted avg       1.00      1.00      1.00     30000



In [37]:
from sklearn.metrics import roc_auc_score

print(
    "ROC-AUC:",
    roc_auc_score(
        y_july,
        july_prob
    )
)

ROC-AUC: 0.9539676460001831


In [38]:
# Actual August -> September churn
august_hv = df[
    df["arpu_8"] >= aug_threshold
].copy()

sep_activity = (
    august_hv["total_og_mou_9"].fillna(0)
    + august_hv["total_ic_mou_9"].fillna(0)
    + august_hv["total_rech_num_9"].fillna(0)
    + august_hv["vol_2g_mb_9"].fillna(0)
    + august_hv["vol_3g_mb_9"].fillna(0)
)

august_hv["actual_churn"] = (
    sep_activity == 0
).astype(int)

print(
    "Actual September churn:",
    august_hv["actual_churn"].mean() * 100,
    "%"
)

print(
    august_hv["actual_churn"].value_counts()
)

Actual September churn: 0.5033333333333333 %
actual_churn
0    29849
1      151
Name: count, dtype: int64
